# Word Similarity and Analogy
:label:`sec_synonyms`

In :numref:`sec_word2vec_pretraining`, 
we trained a word2vec model on a small dataset, 
and applied it
to find semantically similar words 
for an input word.
In practice,
word vectors that are pretrained
on large corpora can be
applied to downstream
natural language processing tasks,
which will be covered later
in :numref:`chap_nlp_app`.
To demonstrate 
semantics of pretrained word vectors
from large corpora in a straightforward way,
let's apply them
in the word similarity and analogy tasks.


In [1]:
import os
import torch
from torch import nn

## Loading Pretrained Word Vectors

Below lists pretrained GloVe embeddings of dimension 50, 100, and 300,
which can be downloaded from the [GloVe website](https://nlp.stanford.edu/projects/glove/).
The pretrained fastText embeddings are available in multiple languages.
Here we consider one English version (300-dimensional "wiki.en") that can be downloaded from the
[fastText website](https://fasttext.cc/).


In [27]:
DATA_URL = "https://d2l-data.s3-accelerate.amazonaws.com/"

DATA_HUB = {
    "glove.6b.50d": (
        DATA_URL + "glove.6B.50d.zip",
        "0b8703943ccdb6eb788e6f091b8946e82231bc4d",
    ),
    "glove.6b.100d": (
        DATA_URL + "glove.6B.100d.zip",
        "cd43bfb07e44e6f27cbcc7bc9ae3d80284fdaf5a",
    ),
    "glove.42b.300d": (
        DATA_URL + "glove.42B.300d.zip",
        "b5116e234e9eb9076672cfeabf5469f3eec904fa",
    ),
    "wiki.en": (
        DATA_URL + "wiki.en.zip",
        "c1816da3821ae9f43899be655002f6c723e91b88",
    ),
}

To load these pretrained GloVe and fastText embeddings, we define the following `TokenEmbedding` class.


In [32]:
# All-in-one, no d2l: DATA_HUB + download/extract + TokenEmbedding
import os
import hashlib
import zipfile
import tarfile
import requests
import torch


def sha1sum(path: str, chunk_size: int = 1 << 20) -> str:
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def download(url: str, dst: str, sha1: str | None = None) -> str:
    os.makedirs(os.path.dirname(dst) or ".", exist_ok=True)

    if not os.path.exists(dst):
        print(f"Downloading {url} -> {dst}")
        with requests.get(url, stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(dst, "wb") as f:
                for c in r.iter_content(chunk_size=1 << 20):
                    if c:
                        f.write(c)

    if sha1 is not None and sha1sum(dst) != sha1:
        raise RuntimeError(
            f"SHA1 mismatch for {dst}. Delete it and re-download."
        )
    return dst

def extract(archive_path: str, out_dir: str) -> None:
    os.makedirs(out_dir, exist_ok=True)
    if archive_path.endswith(".zip"):
        with zipfile.ZipFile(archive_path) as z:
            z.extractall(out_dir)
    elif archive_path.endswith((".tar.gz", ".tgz", ".tar")):
        with tarfile.open(archive_path) as t:
            t.extractall(out_dir)
    else:
        raise ValueError(f"Unsupported archive format: {archive_path}")

def download_extract_by_name(name: str, root: str = "./data") -> str:
    """
    Like d2l.download_extract(name), but no d2l.
    - Downloads archive to root/
    - Extracts to root/name/
    - Returns root/name
    """
    if name not in DATA_HUB:
        raise KeyError(f"{name} not in DATA_HUB. Options: {list(DATA_HUB.keys())}")

    url, sha1 = DATA_HUB[name]
    os.makedirs(root, exist_ok=True)

    archive_path = os.path.join(root, os.path.basename(url))
    download(url, archive_path, sha1=sha1)

    out_dir = os.path.join(root, name)

    # Only extract if we don't already have any extracted content
    needs_extract = (not os.path.exists(out_dir)) or (len(os.listdir(out_dir)) == 0)
    if needs_extract:
        extract(archive_path, out_dir)

    return out_dir

def find_embedding_file(extracted_dir: str) -> str:
    """
    Recursively locate a likely embedding text file.
    Prefers:
      1) vec.txt
      2) any *.txt or *.vec
    """
    best = None
    for cur, _, files in os.walk(extracted_dir):
        if "vec.txt" in files:
            return os.path.join(cur, "vec.txt")
        for fn in files:
            if fn.endswith((".txt", ".vec")):
                best = best or os.path.join(cur, fn)
    if best is None:
        raise FileNotFoundError(
            f"No embedding file found under {extracted_dir}. "
            "Expected vec.txt or a .txt/.vec file."
        )
    return best

class TokenEmbedding:
    """Token embedding loader (D2L-style)."""

    def __init__(self, embedding_name: str, root: str = "./data"):
        self.idx_to_token, self.idx_to_vec = self._load_embedding(embedding_name, root)
        self.unknown_idx = 0
        self.token_to_idx = {tok: i for i, tok in enumerate(self.idx_to_token)}

    def _load_embedding(self, embedding_name: str, root: str):
        data_dir = download_extract_by_name(embedding_name, root=root)
        vec_path = find_embedding_file(data_dir)
        print("Using embedding file:", vec_path)

        idx_to_token = ["<unk>"]
        idx_to_vec = []

        with open(vec_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.rstrip().split()
                # Skip header lines (fastText sometimes has "n d")
                if len(parts) <= 2:
                    continue
                token = parts[0]
                vec = [float(x) for x in parts[1:]]
                idx_to_token.append(token)
                idx_to_vec.append(vec)

        if not idx_to_vec:
            raise RuntimeError(f"No vectors loaded from {vec_path}")

        dim = len(idx_to_vec[0])
        idx_to_vec = [[0.0] * dim] + idx_to_vec  # <unk> vector

        return idx_to_token, torch.tensor(idx_to_vec, dtype=torch.float32)

    def __getitem__(self, tokens):
        if isinstance(tokens, str):
            tokens = [tokens]
        indices = [self.token_to_idx.get(tok, self.unknown_idx) for tok in tokens]
        return self.idx_to_vec[torch.tensor(indices, dtype=torch.long)]

    def __len__(self):
        return len(self.idx_to_token)

Below we load the
50-dimensional GloVe embeddings
(pretrained on a Wikipedia subset).
When creating the `TokenEmbedding` instance,
the specified embedding file has to be downloaded if it
was not yet.


In [30]:
glove_6b50d = TokenEmbedding("glove.6b.50d")
print(len(glove_6b50d), glove_6b50d["the"].shape) 

Using embedding file: ./data/glove.6b.50d/glove.6B.50d/vec.txt
400001 torch.Size([1, 50])


Output the vocabulary size. The vocabulary contains 400000 words (tokens) and a special unknown token.


In [14]:
len(glove_6b50d)

400001

We can get the index of a word in the vocabulary, and vice versa.


In [15]:
glove_6b50d.token_to_idx['beautiful'], glove_6b50d.idx_to_token[3367]

(3367, 'beautiful')

## Applying Pretrained Word Vectors

Using the loaded GloVe vectors,
we will demonstrate their semantics
by applying them
in the following word similarity and analogy tasks.


### Word Similarity

Similar to :numref:`subsec_apply-word-embed`,
in order to find semantically similar words
for an input word
based on cosine similarities between
word vectors,
we implement the following `knn`
($k$-nearest neighbors) function.


In [17]:
def knn(W, x, k):
    # Add 1e-9 for numerical stability
    cos = torch.mv(W, x.reshape(-1,)) / (
        torch.sqrt(torch.sum(W * W, axis=1) + 1e-9) * torch.sqrt((x * x).sum()))
    _, topk = torch.topk(cos, k=k)
    return topk, [cos[int(i)] for i in topk]

Then, we 
search for similar words
using the pretrained word vectors 
from the `TokenEmbedding` instance `embed`.


In [18]:
def get_similar_tokens(query_token, k, embed):
    topk, cos = knn(embed.idx_to_vec, embed[[query_token]], k + 1)
    for i, c in zip(topk[1:], cos[1:]):  # Exclude the input word
        print(f'cosine sim={float(c):.3f}: {embed.idx_to_token[int(i)]}')

The vocabulary of the pretrained word vectors
in `glove_6b50d` contains 400000 words and a special unknown token. 
Excluding the input word and unknown token,
among this vocabulary
let's find 
three most semantically similar words
to word "chip".


In [19]:
get_similar_tokens('chip', 3, glove_6b50d)

cosine sim=0.856: chips
cosine sim=0.749: intel
cosine sim=0.749: electronics


Below outputs similar words
to "baby" and "beautiful".


In [20]:
get_similar_tokens('baby', 3, glove_6b50d)

cosine sim=0.839: babies
cosine sim=0.800: boy
cosine sim=0.792: girl


In [21]:
get_similar_tokens('beautiful', 3, glove_6b50d)

cosine sim=0.921: lovely
cosine sim=0.893: gorgeous
cosine sim=0.830: wonderful


### Word Analogy

Besides finding similar words,
we can also apply word vectors
to word analogy tasks.
For example,
“man”:“woman”::“son”:“daughter”
is the form of a word analogy:
“man” is to “woman” as “son” is to “daughter”.
Specifically,
the word analogy completion task
can be defined as:
for a word analogy 
$a : b :: c : d$, given the first three words $a$, $b$ and $c$, find $d$. 
Denote the vector of word $w$ by $\textrm{vec}(w)$. 
To complete the analogy,
we will find the word 
whose vector is most similar
to the result of $\textrm{vec}(c)+\textrm{vec}(b)-\textrm{vec}(a)$.


In [22]:
def get_analogy(token_a, token_b, token_c, embed):
    vecs = embed[[token_a, token_b, token_c]]
    x = vecs[1] - vecs[0] + vecs[2]
    topk, cos = knn(embed.idx_to_vec, x, 1)
    return embed.idx_to_token[int(topk[0])]  # Remove unknown words

Let's verify the "male-female" analogy using the loaded word vectors.


In [23]:
get_analogy('man', 'woman', 'son', glove_6b50d)

'daughter'

Below completes a
“capital-country” analogy: 
“beijing”:“china”::“tokyo”:“japan”.
This demonstrates 
semantics in the pretrained word vectors.


In [24]:
get_analogy('beijing', 'china', 'tokyo', glove_6b50d)

'japan'

For the
“adjective-superlative adjective” analogy
such as 
“bad”:“worst”::“big”:“biggest”,
we can see that the pretrained word vectors
may capture the syntactic information.


In [25]:
get_analogy('bad', 'worst', 'big', glove_6b50d)

'biggest'

To show the captured notion
of past tense in the pretrained word vectors,
we can test the syntax using the
"present tense-past tense" analogy: “do”:“did”::“go”:“went”.


In [26]:
get_analogy('do', 'did', 'go', glove_6b50d)

'went'

## Summary

* In practice, word vectors that are pretrained on large corpora can be applied to downstream natural language processing tasks.
* Pretrained word vectors can be applied to the word similarity and analogy tasks.

